# 🚀 Synthetic Uber Data Generation

## Enterprise Lakehouse Data Engineering for Agentic AI & Advanced RAG

### 📘 Notebook Purpose

This notebook focuses on generating enterprise-scale synthetic datasets required for building the Uber Enterprise Agentic AI Platform.

The generated datasets will simulate large-scale operational ecosystems including:

* ride operations
* driver activity
* rider activity
* pricing systems
* support operations
* delivery workflows
* fraud events
* telemetry streams

The objective is to engineer scalable, retrieval-friendly, and analytics-ready enterprise datasets capable of supporting:

* advanced RAG systems
* retrieval pipelines
* vector databases
* AI agents
* operational analytics
* graph intelligence
* enterprise AI workflows

This notebook marks the beginning of the practical implementation phase of the platform.

---


# 🏗️ Enterprise Data Generation Strategy

The platform intentionally generates synthetic enterprise-scale datasets instead of relying on small tutorial datasets.

---

# 🚀 Engineering Objectives

The generated datasets are designed to simulate:

* high-volume operational systems
* enterprise transaction patterns
* real-time event ecosystems
* operational analytics workloads
* retrieval-heavy AI systems

---

# 🎯 Key Design Principles

## 1️⃣ Relational Integrity

All datasets will maintain realistic relationships between:

* riders
* drivers
* rides
* pricing zones
* support events
* fraud incidents

---

## 2️⃣ Retrieval-Friendly Structure

The schemas are intentionally designed for:

* semantic retrieval
* metadata filtering
* graph relationships
* hybrid retrieval
* AI analytics

---

## 3️⃣ Lakehouse Compatibility

Datasets will support:

* Bronze/Silver/Gold architecture
* Delta Lake storage
* incremental ingestion
* partitioning
* scalable querying

---

## 4️⃣ Enterprise Metadata

All datasets will include enterprise metadata such as:

* ingestion timestamps
* event timestamps
* partition columns
* audit columns
* source tracking

---

# 🚀 Initial Dataset Scope

The first implementation phase focuses on generating foundational operational datasets:

```text id="dq6q9u"
drivers
riders
cities
pricing_zones
rides
trip_events
```

These foundational datasets will support all downstream AI engineering workflows.

---


# 📦 Dependency Installation

In [0]:
# Install Required Libraries

%pip install faker

# 📦 Environment Initialization

In [0]:
# Core Libraries
import random
import uuid
from datetime import datetime, timedelta

# Data Processing
import pandas as pd
import numpy as np

# Faker for Synthetic Data
from faker import Faker

# PySpark
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Initialize Faker
fake = Faker()

# Initialize Spark
spark = SparkSession.builder.appName(
    "UberEnterpriseAgenticAI"
).getOrCreate()

print("✅ Libraries Loaded Successfully")

# ⚙️ Enterprise Configuration Setup

In [0]:
# ==========================================
# Enterprise Configuration Parameters
# ==========================================

CONFIG = {
    
    # ======================================
    # Environment Configuration
    # ======================================
    
    "catalog": "spark_catalog",
    "schema": "uber_ai",
    
    # ======================================
    # Master Data Volumes
    # ======================================
    
    "num_drivers": 5000,
    "num_riders": 50000,
    "num_cities": 10,
    "num_pricing_zones": 100,
    
    # ======================================
    # Transactional Volumes
    # ======================================
    
    "num_rides": 500000,
    "num_trip_events": 2000000,
    
    # ======================================
    # Date Configuration
    # ======================================
    
    "start_date": "2025-01-01",
    "end_date": "2025-03-31",
    
    # ======================================
    # Partitioning Strategy
    # ======================================
    
    "partition_column": "event_date",
    
    # ======================================
    # Enterprise Metadata
    # ======================================
    
    "created_by": "UberEnterpriseAgenticAI",
    "environment": "dev"
}

print("✅ Enterprise Configuration Loaded")
print(CONFIG)

# 🌍 Generate City Master Data

In [0]:
# ==========================================
# Generate City Master Data
# ==========================================

cities_data = [
    ("HYD", "Hyderabad", "India"),
    ("BLR", "Bangalore", "India"),
    ("MUM", "Mumbai", "India"),
    ("DEL", "Delhi", "India"),
    ("CHE", "Chennai", "India"),
    ("PUN", "Pune", "India"),
    ("KOL", "Kolkata", "India"),
    ("AHM", "Ahmedabad", "India"),
    ("NYC", "New York", "USA"),
    ("SFO", "San Francisco", "USA")
]

cities_schema = StructType([
    StructField("city_id", StringType(), False),
    StructField("city_name", StringType(), False),
    StructField("country", StringType(), False)
])

cities_df = spark.createDataFrame(
    cities_data,
    schema=cities_schema
)

display(cities_df)

print(f"✅ Generated {cities_df.count()} city records")

# 📍 Generate Pricing Zone Master Data

In [0]:
# ==========================================
# Generate Pricing Zones
# ==========================================

pricing_zone_data = []

city_ids = [row["city_id"] for row in cities_df.collect()]

for i in range(CONFIG["num_pricing_zones"]):

    pricing_zone_data.append((
        f"ZONE_{i+1}",
        random.choice(city_ids),
        fake.city_suffix(),
        float(__builtins__.round(random.uniform(1.0, 5.0), 2))
    ))

pricing_zone_schema = StructType([
    StructField("zone_id", StringType(), False),
    StructField("city_id", StringType(), False),
    StructField("zone_name", StringType(), False),
    StructField("base_surge_multiplier", DoubleType(), False)
])

pricing_zones_df = spark.createDataFrame(
    pricing_zone_data,
    schema=pricing_zone_schema
)

display(pricing_zones_df)

print(f"✅ Generated {pricing_zones_df.count()} pricing zones")

# 🚘 Generate Driver Master Data

In [0]:
# ==========================================
# Generate Driver Master Data
# ==========================================

driver_data = []

city_ids = [row["city_id"] for row in cities_df.collect()]

driver_statuses = [
    "ACTIVE",
    "INACTIVE",
    "SUSPENDED"
]

vehicle_types = [
    "SEDAN",
    "SUV",
    "AUTO",
    "BIKE",
    "PREMIUM"
]

for i in range(CONFIG["num_drivers"]):

    driver_data.append((
        f"DRV_{i+1}",
        fake.name(),
        random.choice(city_ids),
        random.choice(vehicle_types),
        random.choice(driver_statuses),
        random.randint(1, 15),
        float(__builtins__.round(random.uniform(3.5, 5.0), 2)),
        fake.date_between(
            start_date='-5y',
            end_date='today'
        ).strftime('%Y-%m-%d'),
        datetime.now()
    ))

driver_schema = StructType([
    
    StructField("driver_id", StringType(), False),
    StructField("driver_name", StringType(), False),
    StructField("city_id", StringType(), False),
    StructField("vehicle_type", StringType(), False),
    StructField("driver_status", StringType(), False),
    StructField("years_experience", IntegerType(), False),
    StructField("driver_rating", DoubleType(), False),
    StructField("joining_date", StringType(), False),
    StructField("created_ts", TimestampType(), False)
])

drivers_df = spark.createDataFrame(
    driver_data,
    schema=driver_schema
)

display(drivers_df)

print(f"✅ Generated {drivers_df.count()} driver records")

# 👤 Generate Rider Master Data

In [0]:
# ==========================================
# Generate Rider Master Data
# ==========================================

rider_data = []

rider_statuses = [
    "ACTIVE",
    "INACTIVE",
    "BLOCKED"
]

for i in range(CONFIG["num_riders"]):

    rider_data.append((
        f"RID_{i+1}",
        fake.name(),
        random.choice(city_ids),
        fake.email(),
        fake.phone_number(),
        random.choice(rider_statuses),
        float(__builtins__.round(random.uniform(3.0, 5.0), 2)),
        fake.date_between(
            start_date='-5y',
            end_date='today'
        ).strftime('%Y-%m-%d'),
        datetime.now()
    ))

rider_schema = StructType([
    
    StructField("rider_id", StringType(), False),
    StructField("rider_name", StringType(), False),
    StructField("city_id", StringType(), False),
    StructField("email", StringType(), False),
    StructField("phone_number", StringType(), False),
    StructField("rider_status", StringType(), False),
    StructField("rider_rating", DoubleType(), False),
    StructField("registration_date", StringType(), False),
    StructField("created_ts", TimestampType(), False)
])

riders_df = spark.createDataFrame(
    rider_data,
    schema=rider_schema
)

display(riders_df)

print(f"✅ Generated {riders_df.count()} rider records")

# 🧠 Cache Reference Data

In [0]:
# ==========================================
# Cache Small Reference Datasets
# ==========================================

city_ids = [
    row["city_id"]
    for row in cities_df.select("city_id").collect()
]

zone_ids = [
    row["zone_id"]
    for row in pricing_zones_df.select("zone_id").collect()
]

driver_ids = [
    row["driver_id"]
    for row in drivers_df.select("driver_id").collect()
]

rider_ids = [
    row["rider_id"]
    for row in riders_df.select("rider_id").collect()
]

print("✅ Reference Data Cached")

# 🚖 Generate Ride Transaction Data

In [0]:
# ==========================================
# Generate Ride Transaction Data
# ==========================================

ride_data = []

ride_statuses = [
    "COMPLETED",
    "CANCELLED",
    "ONGOING"
]

payment_methods = [
    "CREDIT_CARD",
    "UPI",
    "CASH",
    "WALLET"
]

start_date = datetime.strptime(
    CONFIG["start_date"],
    "%Y-%m-%d"
)

end_date = datetime.strptime(
    CONFIG["end_date"],
    "%Y-%m-%d"
)

date_range_days = (end_date - start_date).days

for i in range(CONFIG["num_rides"]):

    ride_timestamp = start_date + timedelta(
        days=random.randint(0, date_range_days),
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59)
    )

    fare_amount = float(
        __builtins__.round(
            random.uniform(50, 2500),
            2
        )
    )

    distance_km = float(
        __builtins__.round(
            random.uniform(1, 40),
            2
        )
    )

    ride_data.append((
        f"RIDE_{i+1}",
        random.choice(rider_ids),
        random.choice(driver_ids),
        random.choice(city_ids),
        random.choice(zone_ids),
        ride_timestamp,
        random.choice(ride_statuses),
        fare_amount,
        distance_km,
        random.randint(5, 90),
        random.choice(payment_methods),
        ride_timestamp.strftime('%Y-%m-%d'),
        datetime.now()
    ))

ride_schema = StructType([

    StructField("ride_id", StringType(), False),
    StructField("rider_id", StringType(), False),
    StructField("driver_id", StringType(), False),
    StructField("city_id", StringType(), False),
    StructField("zone_id", StringType(), False),
    StructField("ride_timestamp", TimestampType(), False),
    StructField("ride_status", StringType(), False),
    StructField("fare_amount", DoubleType(), False),
    StructField("distance_km", DoubleType(), False),
    StructField("ride_duration_minutes", IntegerType(), False),
    StructField("payment_method", StringType(), False),
    StructField("event_date", StringType(), False),
    StructField("created_ts", TimestampType(), False)
])

rides_df = spark.createDataFrame(
    ride_data,
    schema=ride_schema
)

display(rides_df)

print(f"✅ Generated {rides_df.count()} ride records")

# 📡 Generate Trip Event Data

In [0]:
# ==========================================
# Generate Trip Event Data
# ==========================================

trip_event_data = []

event_types = [
    "RIDE_REQUESTED",
    "DRIVER_ASSIGNED",
    "DRIVER_ARRIVED",
    "TRIP_STARTED",
    "TRIP_COMPLETED",
    "PAYMENT_PROCESSED",
    "RIDE_CANCELLED"
]

ride_ids = [
    row["ride_id"]
    for row in rides_df.select("ride_id").limit(100000).collect()
]

for i in range(CONFIG["num_trip_events"]):

    event_timestamp = start_date + timedelta(
        days=random.randint(0, date_range_days),
        hours=random.randint(0, 23),
        minutes=random.randint(0, 59),
        seconds=random.randint(0, 59)
    )

    trip_event_data.append((
        f"EVENT_{i+1}",
        random.choice(ride_ids),
        random.choice(event_types),
        event_timestamp,
        random.choice(city_ids),
        random.choice(zone_ids),
        event_timestamp.strftime('%Y-%m-%d'),
        datetime.now()
    ))

trip_event_schema = StructType([

    StructField("event_id", StringType(), False),
    StructField("ride_id", StringType(), False),
    StructField("event_type", StringType(), False),
    StructField("event_timestamp", TimestampType(), False),
    StructField("city_id", StringType(), False),
    StructField("zone_id", StringType(), False),
    StructField("event_date", StringType(), False),
    StructField("created_ts", TimestampType(), False)
])

trip_events_df = spark.createDataFrame(
    trip_event_data,
    schema=trip_event_schema
)

display(trip_events_df.limit(100))

print(f"✅ Generated {trip_events_df.count()} trip event records")

# 💾 Persist Master & Transactional Data to Delta Lake

In [0]:
# ==========================================
# Define Delta Lake Storage Paths
# ==========================================

BASE_PATH = CONFIG["base_path"]

PATHS = {

    "cities": f"{BASE_PATH}/bronze/cities",
    "pricing_zones": f"{BASE_PATH}/bronze/pricing_zones",
    "drivers": f"{BASE_PATH}/bronze/drivers",
    "riders": f"{BASE_PATH}/bronze/riders",
    "rides": f"{BASE_PATH}/bronze/rides",
    "trip_events": f"{BASE_PATH}/bronze/trip_events"
}

print("✅ Delta Paths Initialized")
print(PATHS)

# 🥉 Write Bronze Delta Tables

In [0]:
# ==========================================
# Persist Bronze Layer Managed Delta Tables
# ==========================================

# Cities
cities_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("uber_ai.cities")

print("✅ cities table created")

# Pricing Zones
pricing_zones_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("uber_ai.pricing_zones")

print("✅ pricing_zones table created")

# Drivers
drivers_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("uber_ai.drivers")

print("✅ drivers table created")

# Riders
riders_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("uber_ai.riders")

print("✅ riders table created")

# Rides
rides_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("event_date") \
    .saveAsTable("uber_ai.rides")

print("✅ rides table created")

# Trip Events
trip_events_df.write \
    .format("delta") \
    .mode("overwrite") \
    .partitionBy("event_date") \
    .saveAsTable("uber_ai.trip_events")

print("✅ trip_events table created")

print("✅ All Bronze Managed Delta Tables Created Successfully")